In [11]:
pip install charset_normalizer

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
pip install regex

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
from pathlib import Path
from charset_normalizer import from_path
import re

1. Write a robust file reader function

In [6]:
def file_reader() -> tuple[str, str]:
    """Read every file under data/2026 and return (file_text_content, encoding_used)."""

    base_dir = Path.cwd() / "data" / "2026"
    files = sorted(p for p in base_dir.rglob("*") if p.is_file())

    texts: list[str] = []
    encodings: list[str] = []

    for f in files:
        raw = f.read_bytes()  # open each file only once

        try:
            texts.append(raw.decode("utf-8-sig"))  # "utf-8-sig" strips a BOM if present
            encodings.append("utf-8")
        except UnicodeDecodeError:
            texts.append(raw.decode("iso-8859-1"))
            encodings.append("iso-8859-1")

    # Preserve order of first appearance, drop duplicates
    encoding_used = ", ".join(dict.fromkeys(encodings))

    return "\n".join(texts), encoding_used

In [7]:
text, encoding = file_reader()

print(encoding)
print(text)

utf-8
[2026-10-15 07:02:36] [POS_01] item=Espresso price=3.34 EUR (SUCCESS)
[2026-10-15 07:03:26] [POS_01] item=Latte price=4.75 EUR (SUCCESS)
[2026-10-15 07:03:43] [POS_01] item=Cappuccino price=4.12 NOK (SUCCESS)
[2026-10-15 07:04:20] [POS_01] item=Latte price=4.75 USD (SUCCESS)
[2026-10-15 07:05:32] [POS_01] item=Cappuccino price=3.80 NOK (REFUND)
[2026-10-15 07:07:28] [POS_01] item=Muffin price=3.84 USD (SUCCESS)
[2026-10-15 07:09:56] [POS_01] item=Sandwich price=7.83 USD (SUCCESS)
[2026-10-15 07:10:30] [POS_01] item=Muffin price=3.16 NOK (SUCCESS)
[2026-10-15 07:13:00] [POS_01] item=Espresso price=3.14 EUR (SUCCESS)
--- PRINTER PAPER LOW ---
[2026-10-15 07:13:55] [POS_01] item=Latte price=4.00 USD (CANCELLED)
[2026-10-15 07:16:22] [POS_01] item=Espresso price=3.15 USD (REFUND)
=== NETWORK SYNC OK ===
[2026-10-15 07:17:56] [POS_01] item=Sandwich price=7.36 EUR (CANCELLED)
[2026-10-15 07:20:54] [POS_01] item=Croissant price=4.30 EUR (REFUND)
[2026-10-15 07:21:34] [POS_01] item=Latte

2. Write a log parser function that uses regular expressions

In [11]:

def parse_log(text: str) -> list[dict]:
    """This function extracts valid sales records from log text, ignoring noise lines."""

    # Define what a valid line should look like
    VALID_LINE = re.compile(
        r"""
        ^\[(?P<timestamp>\d{4}-\d{2}-\d{2}\ \d{2}:\d{2}:\d{2})\]   # [2026-10-15 08:30:00]
        \s+\[(?P<register_id>[A-Za-z0-9_]+)\]                      # [POS_01]
        \s+item=(?P<item>.+?)                                      # Latte (spaces allowed)
        \s+price=(?P<price>\d+(?:\.\d+)?)                          # 4.50
        \s+(?P<currency>[A-Z]{3})                                  # EUR, NOK, USD
        \s+\((?P<status>SUCCESS|REFUND|CANCELLED)\)\s*$            # (SUCCESS)
        """,
        re.VERBOSE,
    )

    # Define list to store the valid sales lines 
    valid_sales_lines = []

    # Iterate over each line, store matching lines and remove the noise
    for line in text.splitlines():
        match = VALID_LINE.match(line.strip())

        if not match:
            continue  # skip the noise

        # Store a dictionary of all the named groups from the matches
        record = match.groupdict()

        # Convert the price into a floa
        record["price"] = float(record["price"])

        # Add record to valid_sales_lines list
        valid_sales_lines.append(record)

    return valid_sales_lines

In [12]:
valid_sales = parse_log(text)

valid_sales

[{'timestamp': '2026-10-15 07:02:36',
  'register_id': 'POS_01',
  'item': 'Espresso',
  'price': 3.34,
  'currency': 'EUR',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:03:26',
  'register_id': 'POS_01',
  'item': 'Latte',
  'price': 4.75,
  'currency': 'EUR',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:03:43',
  'register_id': 'POS_01',
  'item': 'Cappuccino',
  'price': 4.12,
  'currency': 'NOK',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:04:20',
  'register_id': 'POS_01',
  'item': 'Latte',
  'price': 4.75,
  'currency': 'USD',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:05:32',
  'register_id': 'POS_01',
  'item': 'Cappuccino',
  'price': 3.8,
  'currency': 'NOK',
  'status': 'REFUND'},
 {'timestamp': '2026-10-15 07:07:28',
  'register_id': 'POS_01',
  'item': 'Muffin',
  'price': 3.84,
  'currency': 'USD',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:09:56',
  'register_id': 'POS_01',
  'item': 'Sandwich',
  'price': 7.83,
  'curre